# train_density.py

**Updated for Desktop machine:** `/home/pun/Desktop`

**Description:** Train YOLOv11 on Density Detection Dataset - Updated paths for current machine

## Imports

In [1]:
#!/usr/bin/env python3
"""
Train YOLOv11 on Density Variant Dataset
Updated for /home/pun/Desktop
"""
from ultralytics import YOLO
import os
import torch

print("="*70)
print("🚀 YOLOv11 Training on Density Dataset")
print("="*70)

🚀 YOLOv11 Training on Density Dataset


## Configuration

In [2]:
# Paths configuration
BASE_DIR = "/home/pun/Desktop"
DATA_YAML = f"{BASE_DIR}/yolov11/dataset/data_density.yaml"

# Training configuration
MODEL_NAME = 'yolo11n.pt'  # Nano model (fastest)
EPOCHS = 30
ITERATIONS = 200
OPTIMIZER = 'SGD'
IMG_SIZE = 320
BATCH_SIZE = 8
PROJECT_NAME = 'density_tune'

# Device configuration - auto-detect GPU
if torch.cuda.is_available():
    DEVICE = 'cuda'
    print("✅ GPU (CUDA) detected - using GPU acceleration")
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
    print("✅ Apple Silicon (MPS) detected - using MPS acceleration")
else:
    DEVICE = 'cpu'
    print("⚠️  No GPU detected - using CPU (training will be slower)")

print(f"\n📁 Dataset config: {DATA_YAML}")
print(f"🔧 Device: {DEVICE}")
print(f"🏋️  Epochs: {EPOCHS}")
print(f"🔄 Iterations per epoch: {ITERATIONS}")
print(f"⚙️  Optimizer: {OPTIMIZER}")
print(f"📏 Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"📦 Batch size: {BATCH_SIZE}")
print("="*70)

✅ GPU (CUDA) detected - using GPU acceleration

📁 Dataset config: /home/pun/Desktop/yolov11/dataset/data_density.yaml
🔧 Device: cuda
🏋️  Epochs: 30
🔄 Iterations per epoch: 200
⚙️  Optimizer: SGD
📏 Image size: 320x320
📦 Batch size: 8


## Verify Dataset

In [3]:
# Verify dataset exists
if not os.path.exists(DATA_YAML):
    print(f"❌ ERROR: Dataset config not found at {DATA_YAML}")
    print("Please make sure the yolov11/dataset folder is in the correct location.")
else:
    print(f"✅ Dataset config found: {DATA_YAML}")
    
    # Check if images exist
    train_dir = f"{BASE_DIR}/yolov11/dataset/images/train/density"
    val_dir = f"{BASE_DIR}/yolov11/dataset/images/val/density"
    test_dir = f"{BASE_DIR}/yolov11/dataset/images/test/density"
    
    train_count = len([f for f in os.listdir(train_dir) if f.endswith('.jpg')]) if os.path.exists(train_dir) else 0
    val_count = len([f for f in os.listdir(val_dir) if f.endswith('.jpg')]) if os.path.exists(val_dir) else 0
    test_count = len([f for f in os.listdir(test_dir) if f.endswith('.jpg')]) if os.path.exists(test_dir) else 0
    
    print(f"\n📊 Dataset Statistics:")
    print(f"   Training images: {train_count}")
    print(f"   Validation images: {val_count}")
    print(f"   Test images: {test_count}")
    print(f"   Total images: {train_count + val_count + test_count}")

✅ Dataset config found: /home/pun/Desktop/yolov11/dataset/data_density.yaml

📊 Dataset Statistics:
   Training images: 192
   Validation images: 24
   Test images: 72
   Total images: 288


## Initialize Model

In [4]:
# Initialize YOLO model - will auto-download if not present
print("\n🔧 Initializing YOLO model...")
model = YOLO(MODEL_NAME)
print(f"✅ Model loaded: {MODEL_NAME}")


🔧 Initializing YOLO model...
✅ Model loaded: yolo11n.pt


## Start Tuning

In [5]:
# Train the model
print("\n" + "="*70)
print("🏋️  Starting Tuning...")
print("="*70 + "\n")

search_space = {
    # Optimization parameters
    "lr0": (1e-5, 1e-1),            # Initial learning rate
    "lrf": (0.01, 1.0),             # Final learning rate factor
    "momentum": (0.60, 0.98),       # SGD momentum
    "weight_decay": (0.0, 0.001),   # Weight decay
    "warmup_epochs": (0.0, 5.0),    # Warmup epochs
    "close_mosaic": (0.0, 20.0),    # Epochs to disable mosaic
    
    # Preserve grayscale/density values
    "hsv_h": (0.0, 0.1),            # Hue augmentation
    "hsv_s": (0.0, 0.9),            # Saturation augmentation
    "hsv_v": (0.0, 0.9),            # Value brightness augmentation

    # Augmentation parameters - optimized for density detection
    "degrees": (0.0, 45.0),         # Rotation augmentation
    "translate": (0.0, 0.9),        # Translation augmentation
    "scale": (0.0, 0.9),            # Scale augmentation
    "flipud": (0.0, 1.0),           # Vertical flip probability
    "fliplr": (0.0, 1.0),           # Horizontal flip probability
    "mosaic": (0.0, 1.0),           # Mosaic augmentation probability
    "mixup": (0.0, 1.0),            # Mixup
}

results = model.tune(
    # Dataset
    data=DATA_YAML,
    
    # Training parameters
    epochs=EPOCHS,
    iterations=ITERATIONS,
    optimizer=OPTIMIZER,
    space=search_space,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,

    # Set initial guess for 0 default parameters values
    degrees=20.0,
    flipud=0.5,
    mixup=0.1,

    # Output
    name=PROJECT_NAME,
    project='runs/detect',
    
    # Device
    device=DEVICE,
    workers=4,

    # Output options
    save=False,
    plots=False,
    verbose=False,
    val=False
)


🏋️  Starting Tuning...

Tuner: Initialized Tuner instance with 'tune_dir=/home/pun/Desktop/notebooks_density/training/runs/detect/density_tune3'
Tuner: 💡 Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/200 with hyperparameters: {'lr0': 0.01, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'close_mosaic': 10, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'degrees': 20.0, 'translate': 0.1, 'scale': 0.5, 'flipud': 0.5, 'fliplr': 0.5, 'mosaic': 1.0, 'mixup': 0.1}
Ultralytics 8.3.241 🚀 Python-3.12.12 torch-2.9.1+cu128 CUDA:0 (NVIDIA GeForce RTX 4090, 24202MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/pun/Desktop/yolov11/dataset/data_density.yaml, degrees=20.0, determinis

## Tuning Results

In [6]:
print(f"\n{'='*70}")
print("✅ Tuning Complete!")
print(f"{'='*70}")
print(f"\n📁 Results saved to: runs/detect/{PROJECT_NAME}/")
print(f"🏆 Best hyperparameters: runs/detect/{PROJECT_NAME}/best_hyperparameters.yaml")
print(f"\n📈 Tuning metrics and plots saved in the results folder.")
print(f"{'='*70}")


✅ Tuning Complete!

📁 Results saved to: runs/detect/density_tune/
🏆 Best hyperparameters: runs/detect/density_tune/best_hyperparameters.yaml

📈 Tuning metrics and plots saved in the results folder.
